# Session 3: Classes, Dataclasses, and Training Configuration

**Focus:** syntax first. **Time:** 60-90 minutes.

Read each syntax breakdown, predict, run, modify, then build. Do not open stronger hints until you have attempted the TODO.

## 0. Quick concept check

Answer from memory. These answers decide whether any concept needs revisiting.

In [ ]:
concept_check = {
    "when_class_is_justified": "when related data must stay together across operations",
    "class_vs_instance": "class is a blueprint which describes the data stored, while instance is the actual object of a class",
    "why_config_values_belong_together": " they describe one training run and should be passed and validated as one unit.",
}

## 1. Normal class syntax

A class defines a reusable object structure. Calling the class creates an **instance** that stores its own attribute values.

### Complete non-ML example

In [1]:
class GameSettings:
    def __init__(self, difficulty: str, sound_enabled: bool = True) -> None:
        self.difficulty = difficulty
        self.sound_enabled = sound_enabled


player_one_settings = GameSettings("hard", sound_enabled=False)
print(player_one_settings.difficulty)
print(player_one_settings.sound_enabled)

hard
False


### Syntax anatomy

```python
class ClassName:
    def __init__(self, parameter: type, optional: type = default) -> None:
        self.attribute = parameter
        self.optional = optional

instance = ClassName(argument, optional=value)
instance.attribute
```

- `class GameSettings:` starts the class definition.
- `__init__` runs when `GameSettings(...)` creates an instance.
- `self` is the instance currently being initialized. Python supplies it automatically during the class call.
- `self.difficulty = difficulty` stores the parameter value on that instance.
- `__init__` returns `None`; the class call returns the new `GameSettings` instance.
- Valid values are part of your contract. Type hints alone do not enforce them.

### Predict, then run

Predict the two printed values and the type returned by the class call.

In [ ]:
normal_class_prediction = {
    "first_line": "medium",
    "second_line": "true",
    "type_of_instance": "GameSettings",
}

In [3]:
settings = GameSettings("medium")
print(settings.difficulty)
print(settings.sound_enabled)
print(type(settings))

medium
True
<class '__main__.GameSettings'>


## 2. Copy and modify: normal class

Complete the assignments. `username` is required; `dark_mode` defaults to `False`.

Parameters: `username: str`, `dark_mode: bool = False`. `__init__` returns `None`; `UserPreferences(...)` returns a `UserPreferences` instance.

In [7]:
class UserPreferences:
    def __init__(self, username: str, dark_mode: bool = False) -> None:
        # TODO: store both parameter values as instance attributes.
        if username:
            self.username = username
            self.dark_mode = dark_mode
        else:
            raise ValueError("user name is required")

<details><summary>Stronger hint</summary>Inside `__init__`, use the pattern `self.attribute_name = parameter_name` once for each parameter.</details>

In [8]:
# CHECK
preferences = UserPreferences("umer")
assert preferences.username == "umer"
assert preferences.dark_mode is False
dark_preferences = UserPreferences("umer", dark_mode=True)
assert dark_preferences.dark_mode is True
print("Passed: normal class construction and defaults")

Passed: normal class construction and defaults


## 3. Dataclass syntax

A dataclass is useful when the class mainly stores related data. `@dataclass` generates repetitive methods such as `__init__`, readable representation, and equality comparison.

### Complete non-ML example

In [9]:
from dataclasses import dataclass


@dataclass
class TripSettings:
    destination: str
    days: int = 1


trip = TripSettings("Lahore", days=3)
print(trip)
print(trip.destination)
print(trip == TripSettings("Lahore", days=3))

TripSettings(destination='Lahore', days=3)
Lahore
True


### Syntax anatomy

```python
from dataclasses import dataclass

@dataclass
class ClassName:
    required_field: type
    optional_field: type = default
```

- `@dataclass` modifies the class directly below it.
- Field annotations become generated constructor parameters.
- Required fields must appear before fields with defaults.
- `TripSettings(...)` returns a `TripSettings` instance.
- `trip.destination` returns a `str`; `trip.days` returns an `int`.

Common syntax error: putting a required field after a default field causes `TypeError: non-default argument ... follows default argument`.

## 4. Partially completed dataclass

Complete `DownloadSettings`. `folder` is required. `retry_count` defaults to `3`.

In [10]:
@dataclass
class DownloadSettings:
    # TODO: add the required typed folder field.
    folder: str
    # TODO: add the typed retry_count field with its default.
    retry_count: int = 3

In [11]:
# CHECK
download = DownloadSettings("datasets")
assert download.folder == "datasets"
assert download.retry_count == 3
assert DownloadSettings("datasets") == DownloadSettings("datasets")
print("Passed: dataclass fields, default, and generated equality")

Passed: dataclass fields, default, and generated equality


## 5. Validation with `__post_init__`

The generated `__init__` assigns all fields first. Dataclasses then call `__post_init__(self)`, making it the normal place for validation.

`@dataclass(frozen=True)` prevents ordinary attribute reassignment after construction. It is useful for configuration that should remain stable during a run.

In [14]:
@dataclass(frozen=True)
class TicketOrder:
    ticket_price: float
    quantity: int = 1

    def __post_init__(self) -> None:
        if self.ticket_price <= 0:
            raise ValueError("ticket_price must be greater than zero")
        if self.quantity <= 0:
            raise ValueError("quantity must be greater than zero")


order = TicketOrder(1500.0, quantity=2)
print(order)

TicketOrder(ticket_price=1500.0, quantity=2)


### Parameters, returns, and failure behavior

- `ticket_price: float`: required, valid when greater than zero.
- `quantity: int = 1`: optional, valid when greater than zero.
- `__post_init__` returns `None` when validation passes.
- `TicketOrder(...)` returns a validated `TicketOrder` instance or raises `ValueError`.
- Reassigning `order.quantity` raises `FrozenInstanceError`.

Frozen does not make every possible nested mutable object immutable. This configuration uses simple immutable field values, so that limitation does not affect it.

## 6. Independent ML task: `TrainingConfig`

Create an immutable training configuration.

Required contract:

- `learning_rate: float`: required and greater than zero.
- `epochs: int`: required and greater than zero.
- `threshold: float = 0.5`: must be between `0.0` and `1.0`, inclusive.
- Construction returns a validated `TrainingConfig` instance.
- Invalid values raise `ValueError`.
- Fields cannot be reassigned after construction.

Syntax hints: use `@dataclass(frozen=True)`, declare required fields before the default field, then define `__post_init__(self) -> None`.

In [33]:
# TODO: add the appropriate dataclass decorator.
@dataclass(frozen = True)
class TrainingConfig:
    # TODO: declare the three typed fields in valid order.
    learning_rate: float
    epochs: int
    threshold: float = 0.5
    # TODO: define __post_init__ and validate all three fields.
    def __post_init__(self) -> None:
        if self.learning_rate <= 0:
            raise ValueError("Learning rate should be greater than zero")
        if self.epochs <= 0:
            raise ValueError("Epochs should be greater than zero")
        if self.threshold < 0.00 or self.threshold > 1.0:
            raise ValueError("Threshold should be in range 0 and 1")



<details><summary>Stronger hint</summary>Use the `TicketOrder` structure, but write three field declarations. Add one invalid-condition check for each field. Threshold has two invalid sides joined with `or`.</details>

In [31]:
# SETUP: run this helper. Understanding every line is not required yet.
def assert_raises(expected_exception: type[Exception], function, *args, **kwargs) -> None:
    try:
        function(*args, **kwargs)
    except expected_exception:
        return
    raise AssertionError(f"Expected {expected_exception.__name__}")

In [34]:
# AUTOMATED CHECKS
from dataclasses import FrozenInstanceError, is_dataclass

config = TrainingConfig(learning_rate=0.01, epochs=50)
assert is_dataclass(config)
assert config.learning_rate == 0.01
assert config.epochs == 50
assert config.threshold == 0.5
assert TrainingConfig(0.01, 50) == TrainingConfig(0.01, 50)
assert TrainingConfig(0.01, 50, threshold=0.0).threshold == 0.0
assert TrainingConfig(0.01, 50, threshold=1.0).threshold == 1.0
assert_raises(ValueError, TrainingConfig, 0.0, 50)
assert_raises(ValueError, TrainingConfig, -0.01, 50)
assert_raises(ValueError, TrainingConfig, 0.01, 0)
assert_raises(ValueError, TrainingConfig, 0.01, -2)
assert_raises(ValueError, TrainingConfig, 0.01, 50, threshold=-0.1)
assert_raises(ValueError, TrainingConfig, 0.01, 50, threshold=1.1)
assert_raises(FrozenInstanceError, setattr, config, "epochs", 100)
print("Passed: construction, defaults, validation, equality, and immutability")

Passed: construction, defaults, validation, equality, and immutability


## 7. Dataclass versus dictionary

A dictionary is flexible, but its expected keys, defaults, and validation are not built into the object. A dataclass gives the configuration one explicit shape and a predictable interface.

Do not conclude that classes are always better. A short temporary mapping may still be simpler as a dictionary. This class earns its place because the same validated configuration will move through training modules.

In [ ]:
dictionary_config = {"learning_rate": 0.01, "epochs": 50, "threshhold": 0.5}
dataclass_config = TrainingConfig(learning_rate=0.01, epochs=50, threshold=0.5)

comparison = {
    "dictionary_typo_consequence": "the misspelled key silently becomes a different key. Mutation is a separate issue.",
    "dataclass_interface_advantage": "we get one shape and predictable interface",
    "why_training_config_earns_a_class": "because that same validated configuration values will be used in the training",
    "when_dictionary_is_simpler": "when we have to make short temporary mappings",
}

## 8. Common syntax mistakes

| Mistake | Typical result | Correction |
|---|---|---|
| Forgetting `self` in a method | Missing/extra argument `TypeError` | Put `self` first |
| Writing `attribute = value` instead of `self.attribute = value` | Attribute is not stored on the instance | Assign through `self` |
| Required dataclass field after a default field | `non-default argument follows default argument` | Required fields first |
| Returning a value from `__init__` | `__init__() should return None` | Store attributes; do not return the instance |
| Assigning to a frozen field | `FrozenInstanceError` | Create a new configuration instance |
| Expecting annotations to validate values | Invalid value may be accepted | Validate in `__post_init__` |

## 9. Explain your code

Use full sentences. Explain syntax and behavior, not whether something was merely `good`.

In [ ]:
explanation = {
    "what_self_refers_to": "the current instance of a class",
    "init_return_vs_class_call_return": "init doest not return anything while class call returns a instance",
    "what_dataclass_generates": "it generates methods such as __init__, __repr__, and __eq__",
    "why_required_fields_come_first": "Python cannot place a required parameter after a default parameter in the generated constructor",
    "when_post_init_runs": "after the values are assigned to the data variables",
    "what_frozen_true_changes": "it blocks attribute reassignment",
    "training_config_return_type": "TrainingConfig class object",
}

most_difficult_syntax = ""
help_used = ""
one_question_i_still_have = ""

## Compact syntax reference

```python
class NormalClass:
    def __init__(self, value: float) -> None:
        self.value = value

instance = NormalClass(1.0)
instance.value

from dataclasses import dataclass

@dataclass(frozen=True)
class Config:
    required: float
    optional: int = 10

    def __post_init__(self) -> None:
        if self.required <= 0:
            raise ValueError("required must be positive")
```

## Retrieval task: complete 2-3 days later

Without reopening this notebook, rebuild an immutable two-field dataclass with one default and one validation rule. Then explain what `self`, `@dataclass`, and `__post_init__` do.

In [ ]:
retrieval_date = ""
retrieval_code = ""
retrieval_explanation = ""

## Save and stop

Save after all immediate checks pass. Do not complete the retrieval task today. Send the notebook for review before committing.